In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (roc_auc_score, confusion_matrix,
                             precision_recall_curve)
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from imblearn.under_sampling import NearMiss
import joblib
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================================================================
# Project paths
# ============================================================================
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODEL_DIR = RESULTS_DIR / 'model'
TABLE_DIR = RESULTS_DIR / 'table'
FIGURE_DIR = RESULTS_DIR / 'figures'

for d in [DATA_DIR, MODEL_DIR, TABLE_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project paths ready")

# ============================================================================
# Utility Functions
# ============================================================================
def find_optimal_threshold(y_true, y_proba):
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    with np.errstate(divide='ignore', invalid='ignore'):
        f1_scores = 2 * (precision * recall) / (precision + recall)
    f1_scores = np.nan_to_num(f1_scores)
    if len(f1_scores) == 0:
        return 0.5, 0.0
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5
    return optimal_threshold, f1_scores[optimal_idx]

def evaluate_model(model, X_test, y_test, model_name):
    y_prob = model.predict_proba(X_test)[:, 1]
    threshold, f1 = find_optimal_threshold(y_test, y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    auc = roc_auc_score(y_test, y_prob)
    cm = confusion_matrix(y_test, y_pred)
    tp = cm[1, 1]
    total_bankrupt = cm[1, 0] + cm[1, 1]
    recall = tp / total_bankrupt if total_bankrupt > 0 else 0.0

    print(f"\n[{model_name}] Results")
    print(f"AUC: {auc:.4f}")
    print(f"Optimal Threshold: {threshold:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"Correctly predicted bankruptcy (TP): {tp} / {total_bankrupt}  (Recall: {recall:.4f})")

    return {'Model': model_name, 'Threshold': threshold, 'AUC': auc,
            'F1': f1, 'Recall': recall, 'TP': tp, 'Total_Bankrupt_Test': total_bankrupt}

# ============================================================================
# NOTE ON METHODOLOGY (documented here, and to be cited in Section 5.4):
# NearMiss is applied to the FULL dataset BEFORE train/test splitting, matching
# the original paper's approach. This means the held-out test set is also
# NearMiss-balanced and does NOT reflect the original ~1:70 class ratio.
# A leakage-corrected variant (split first, NearMiss on train only) was tested
# and showed the synthetic dataset's minority-class signal does not generalize
# to an untouched, original-ratio test set (test AUC ~0.5). This is treated as
# a limitation of the synthetic data's fidelity, not of the recourse framework,
# and is disclosed explicitly in the paper's Limitations section.
# ============================================================================

# ============================================================================
# 1. Data Loading
# ============================================================================
print("=" * 80)
print("1. Data Loading (full corpus, ratio features, cleaned)")
print("=" * 80)

df = pd.read_csv(DATA_DIR / 'selected_data_for_modeling_full_ratio_clean.csv')
feature_cols = joblib.load(DATA_DIR / 'feature_names_full_ratio_clean.pkl')

X = df[feature_cols]
y = df['PERF_12M']

print(f"Full dataset shape: {X.shape}")
print(f"Total bankruptcy cases: {y.sum()} ({(y.sum()/len(y))*100:.2f}%)")

# ============================================================================
# 2. NearMiss Undersampling — applied to FULL dataset (paper methodology)
# ============================================================================
print("\n" + "=" * 80)
print("2. Applying NearMiss Undersampling (1:1 ratio, applied before split)")
print("=" * 80)

nm = NearMiss(version=1, n_neighbors=3)
X_resampled, y_resampled = nm.fit_resample(X, y)

print(f"Dataset shape after resampling: {X_resampled.shape}")
print(f"Bankruptcy cases after resampling: {y_resampled.sum()}")
print(f"Solvent cases after resampling: {(y_resampled==0).sum()}")
print("Class ratio adjusted to 1:1.")

df_resampled = pd.DataFrame(X_resampled, columns=X.columns)
df_resampled['PERF_12M'] = y_resampled
df_resampled.to_csv(DATA_DIR / 'resampled_data_final_full.csv', index=False, encoding='utf-8-sig')
print(">> 'resampled_data_final_full.csv' saved (for visualisation)")

# Train/Test Split (80:20) — AFTER resampling, per paper methodology
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

pos_weight = 1.0

# ============================================================================
# 3. Base Model Training (Optuna)
# ============================================================================
print("\n" + "=" * 80)
print("3. XGBoost Training with Optuna Hyperparameter Optimisation")
print("=" * 80)

def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        'random_state': 42,
        'n_jobs': -1,
        'scale_pos_weight': pos_weight,
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7)
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for tr_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, verbose=False)
        scores.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))

    return np.mean(scores)

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=30)

best_params = study.best_params
best_params.update({'random_state': 42, 'n_jobs': -1, 'scale_pos_weight': pos_weight})

print(f"Best CV AUC: {study.best_value:.4f}")

base_model = xgb.XGBClassifier(**best_params)
base_model.fit(X_train, y_train)

res = evaluate_model(base_model, X_test, y_test, 'Final Model (full corpus, ratio features)')

# ============================================================================
# 4. Save Outputs (required for Agent steps)
# ============================================================================
X_test.to_csv(DATA_DIR / 'X_test_final_full.csv', index=False)
y_test.to_csv(DATA_DIR / 'y_test_final_full.csv', index=False)

joblib.dump(base_model, MODEL_DIR / 'base_model_final_full.pkl')
joblib.dump(res['Threshold'], MODEL_DIR / 'base_model_threshold_final_full.pkl')
joblib.dump(list(X.columns), MODEL_DIR / 'selected_features_final_full.pkl')

summary_df = pd.DataFrame([{
    'n_total': len(X), 'n_bankrupt_total': int(y.sum()),
    'n_resampled': len(X_resampled), 'cv_auc': study.best_value,
    'test_auc': res['AUC'], 'test_f1': res['F1'], 'test_recall': res['Recall'],
    'test_tp': res['TP'], 'test_total_bankrupt': res['Total_Bankrupt_Test']
}])
summary_df.to_csv(TABLE_DIR / 'step2_model_summary_full_final.csv', index=False, encoding='utf-8-sig')

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print(summary_df[['n_total', 'n_bankrupt_total', 'n_resampled', 'cv_auc',
                   'test_auc', 'test_f1', 'test_recall']].to_string(index=False))

print("\nAll files saved. Proceed to the next step (DiCE).")

C:\Users\miy\miniconda3\envs\diceml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project paths ready
1. Data Loading (full corpus, ratio features, cleaned)
Full dataset shape: (147724, 62)
Total bankruptcy cases: 2070 (1.40%)

2. Applying NearMiss Undersampling (1:1 ratio, applied before split)
Dataset shape after resampling: (4140, 62)
Bankruptcy cases after resampling: 2070
Solvent cases after resampling: 2070
Class ratio adjusted to 1:1.
>> 'resampled_data_final_full.csv' saved (for visualisation)

3. XGBoost Training with Optuna Hyperparameter Optimisation
Best CV AUC: 0.9427

[Final Model (full corpus, ratio features)] Results
AUC: 0.9390
Optimal Threshold: 0.4683
F1 Score: 0.8673
Confusion Matrix:
[[380  34]
 [ 71 343]]
Correctly predicted bankruptcy (TP): 343 / 414  (Recall: 0.8285)

FINAL SUMMARY
 n_total  n_bankrupt_total  n_resampled   cv_auc  test_auc  test_f1  test_recall
  147724              2070         4140 0.942706  0.939018 0.867257     0.828502

All files saved. Proceed to the next step (DiCE).
